In [2]:
import pandas as pd
import numpy as np

In [3]:
trades = pd.read_excel(r"C:\Keshav S\Investements Operations Platform\Trades.xlsx")

positions = pd.read_excel(r"C:\Keshav S\Investements Operations Platform\Closing_Positions.xlsx")

prices = pd.read_excel(r"C:\Keshav S\Investements Operations Platform\Prices.xlsx")

In [4]:
avg_price = (
    trades.assign(notional=trades["Quantity"] * trades["Trade_Price"])
    .groupby("Security")
    .agg(
        total_notional=("notional", "sum"),
        total_qty=("Quantity", "sum")
    )
    .reset_index()
)

avg_price["Avg_Price"] = avg_price["total_notional"] / avg_price["total_qty"]

avg_price = avg_price[["Security", "Avg_Price"]]

In [5]:
pnl = positions.merge(avg_price, on="Security", how="left")
pnl = pnl.merge(prices, on="Security", how="left")

In [6]:
pnl["Position_Cost"] = pnl["Avg_Price"] * pnl["Closing_Position"]

pnl["Market_Value"] = pnl["Market_Price"] * pnl["Closing_Position"]

pnl["Unrealized_PnL"] = pnl["Market_Value"] - pnl["Position_Cost"]

In [7]:
pnl["PnL_%"] = np.where(
    pnl["Position_Cost"] != 0,
    (pnl["Unrealized_PnL"] / pnl["Position_Cost"]) * 100,
    0
)

In [8]:
pnl_report = pnl[[
    "Security",
    "Opening_Position",
    "Trade_Impact",
    "Closing_Position",
    "Avg_Price",
    "Market_Price",
    "Position_Cost",
    "Market_Value",
    "Unrealized_PnL",
    "PnL_%"
]]

In [9]:
pnl_report.to_excel(
    r"C:\Keshav S\Investements Operations Platform\PnL_Report.xlsx",
    index=False
)